In [1]:
import pandas as pd

cols = ["TRPTRANS", "WHYTO", "R_AGE", "R_SEX", "HHFAMINC",
        "TRPMILES", "HHVEHCNT", "URBRUR"]

trips = pd.read_csv("trippub.csv", usecols=cols)

print(trips.shape)
trips.head()

(923572, 8)


,TRPMILES,TRPTRANS,WHYTO,HHVEHCNT,HHFAMINC,URBRUR,R_AGE,R_SEX
0,5.244,3,19,5,7,1,67,2
1,5.149,3,1,5,7,1,67,2
2,84.004,6,1,5,7,1,66,1
3,81.628,6,3,5,7,1,66,1
4,2.250,3,8,5,7,1,28,2


In [2]:
school = trips[(trips["WHYTO"] == 8) &
               (trips["R_AGE"] >= 18) &
               (trips["R_AGE"] <= 30)]

print(school.shape)
school["TRPTRANS"].value_counts()

(3547, 8)


,count
TRPTRANS,
3,1651
1,688
4,428
6,206
11,162
10,123
2,97
5,76
16,45


In [3]:
# Target Column
mode_map = {
    3: "Car", 4: "Car", 5: "Car", 6: "Car", 8: "Car", 18: "Car",
    1: "Walk",
    2: "Bike",
    10: "Transit", 11: "Transit", 12: "Transit", 13: "Transit",
    14: "Transit", 15: "Transit", 16: "Transit",
}

school = school.copy()
school["MODE"] = school["TRPTRANS"].map(mode_map)
school = school.dropna(subset=["MODE"])

print(school.shape)
school["MODE"].value_counts()

(3523, 9)


,count
MODE,
Car,2372
Walk,688
Transit,366
Bike,97


In [4]:
# Missing value check
features = ["TRPMILES", "HHVEHCNT", "HHFAMINC", "URBRUR", "R_AGE", "R_SEX"]

(school[features] < 0).sum()

,0
TRPMILES,4
HHVEHCNT,0
HHFAMINC,48
URBRUR,0
R_AGE,0
R_SEX,2


In [5]:
# Dropping rows with missing codes
school = school[(school[features] >= 0).all(axis=1)]

print(school.shape)
school["MODE"].value_counts()

(3469, 9)


,count
MODE,
Car,2339
Walk,675
Transit,359
Bike,96


In [6]:
# Split data into training and test sets
from sklearn.model_selection import train_test_split

X = school[features]
y = school["MODE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
y_test.value_counts()

(2775, 6) (694, 6)


,count
MODE,
Car,468
Walk,135
Transit,72
Bike,19


In [7]:
# Train logistic regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000)
)

logreg.fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

        Bike       0.11      0.74      0.19        19
         Car       0.95      0.55      0.70       468
     Transit       0.22      0.43      0.29        72
        Walk       0.55      0.62      0.59       135

    accuracy                           0.56       694
   macro avg       0.46      0.59      0.44       694
weighted avg       0.77      0.56      0.62       694



In [8]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print(classification_report(y_test, y_pred_tree))

              precision    recall  f1-score   support

        Bike       0.09      0.68      0.16        19
         Car       0.93      0.71      0.81       468
     Transit       0.35      0.39      0.37        72
        Walk       0.80      0.64      0.71       135

    accuracy                           0.66       694
   macro avg       0.54      0.61      0.51       694
weighted avg       0.82      0.66      0.72       694



In [9]:
print(trips.shape)
print(school.shape)
print(X_train.shape, X_test.shape)
print(school["R_AGE"].min(), school["R_AGE"].max())

(923572, 8)
(3469, 9)
(2775, 6) (694, 6)
18 30


In [10]:
print(school.shape)
print(school["MODE"].value_counts())
print(school["MODE"].value_counts(normalize=True).round(3))

(3469, 9)
MODE
Car        2339
Walk        675
Transit     359
Bike         96
Name: count, dtype: int64
MODE
Car        0.674
Walk       0.195
Transit    0.103
Bike       0.028
Name: proportion, dtype: float64


In [11]:
from sklearn.metrics import accuracy_score, f1_score

test = X_test.copy()
test["actual"] = y_test
test["pred"] = y_pred_tree

for sex, group in test.groupby("R_SEX"):
    acc = accuracy_score(group["actual"], group["pred"])
    f1 = f1_score(group["actual"], group["pred"], average="macro")
    print(sex, len(group), round(acc, 3), round(f1, 3))

1 335 0.621 0.467
2 359 0.702 0.525


In [12]:
test["INCOME_GRP"] = pd.cut(
    test["HHFAMINC"],
    bins=[0, 5, 8, 11],
    labels=["Low", "Middle", "High"]
)

for grp, group in test.groupby("INCOME_GRP", observed=True):
    acc = accuracy_score(group["actual"], group["pred"])
    f1 = f1_score(group["actual"], group["pred"], average="macro")
    print(grp, len(group), round(acc, 3), round(f1, 3))

Low 271 0.542 0.467
Middle 268 0.731 0.523
High 155 0.755 0.473


In [13]:
print(pd.crosstab(test["INCOME_GRP"], test["actual"], normalize="index").round(2))
print()
print(pd.crosstab(test["R_SEX"], test["actual"], normalize="index").round(2))

actual      Bike   Car  Transit  Walk
INCOME_GRP                           
Low         0.03  0.63     0.13  0.21
Middle      0.01  0.68     0.10  0.20
High        0.04  0.73     0.06  0.17

actual  Bike   Car  Transit  Walk
R_SEX                            
1       0.04  0.65     0.12  0.18
2       0.01  0.69     0.09  0.21


In [14]:
for grp, group in test.groupby("INCOME_GRP", observed=True):
    car = group[group["actual"] == "Car"]
    recall = (car["pred"] == "Car").mean()
    print(grp, len(car), round(recall, 3))

Low 172 0.535
Middle 183 0.809
High 113 0.823


In [15]:
nonsensitive = ["TRPMILES", "HHVEHCNT", "URBRUR"]

tree_ns = DecisionTreeClassifier(
    max_depth=5, class_weight="balanced", random_state=42
)
tree_ns.fit(X_train[nonsensitive], y_train)
test["pred_ns"] = tree_ns.predict(X_test[nonsensitive])

print(classification_report(test["actual"], test["pred_ns"]))

for grp, group in test.groupby("INCOME_GRP", observed=True):
    acc = accuracy_score(group["actual"], group["pred_ns"])
    f1 = f1_score(group["actual"], group["pred_ns"], average="macro")
    print(grp, len(group), round(acc, 3), round(f1, 3))

              precision    recall  f1-score   support

        Bike       0.10      0.68      0.17        19
         Car       0.91      0.65      0.76       468
     Transit       0.23      0.36      0.28        72
        Walk       0.85      0.75      0.80       135

    accuracy                           0.64       694
   macro avg       0.52      0.61      0.50       694
weighted avg       0.81      0.64      0.70       694

Low 271 0.561 0.49
Middle 268 0.69 0.495
High 155 0.684 0.468
